In [396]:
import numpy as np 
import pandas as pd 
from sklearn.preprocessing import FunctionTransformer  , StandardScaler , TargetEncoder
from sklearn.model_selection import KFold  , StratifiedKFold 
from sklearn.base import BaseEstimator , TransformerMixin 
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline 
import lightgbm as lgm 
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import RandomForestClassifier
import catboost as cb 
from sklearn.metrics import balanced_accuracy_score ,accuracy_score
from sklearn.linear_model import LogisticRegression


In [318]:
# load the data 
train_data = pd.read_csv('irr_train.csv' , index_col='id') 
test_data = pd.read_csv('irr_test.csv' , index_col='id')

In [319]:

CAT_COLS = ['Soil_Type' , 'Crop_Type' ,'Crop_Growth_Stage' ,'Crop_Growth_Stage' , 'Season' ,'Irrigation_Type' ,'Water_Source' ,'Mulching_Used' ,'Region' ,'Irrigation_Need' ]
TARGET = 'Irrigation_Need'
NUM_COLS = ['Soil_pH' ,'Soil_Moisture' ,'Organic_Carbon' ,'Electrical_Conductivity' ,'Temperature_C' ,'Humidity' , 'Rainfall_mm' ,'Sunlight_Hours' ,'Wind_Speed_kmh' ,'Field_Area_hectare' , 'Previous_Irrigation_mm']
COLUMNS = test_data.columns.to_list()

In [270]:
for col in COLUMNS : 
    print(train_data.groupby(col)[col].value_counts()) 
    print('--------------------------')
print(train_data.groupby(TARGET)[TARGET].value_counts()) 

Soil_Type
Clay     5059
Loamy    4969
Sandy    5244
Silt     4728
Name: count, dtype: int64
--------------------------
Soil_pH
4.80     9
4.81    17
4.82    15
4.83    51
4.84    23
        ..
8.16    21
8.17    12
8.18    17
8.19    27
8.20    13
Name: count, Length: 341, dtype: int64
--------------------------
Soil_Moisture
8.02     11
8.03      5
8.04      1
8.05      4
8.06      9
         ..
64.90     4
64.93    10
64.96     1
64.97     4
64.99     4
Name: count, Length: 3623, dtype: int64
--------------------------
Organic_Carbon
0.30      1
0.31    159
0.32     61
0.33    223
0.34    225
       ... 
1.56     97
1.57     91
1.58    135
1.59    165
1.60     19
Name: count, Length: 131, dtype: int64
--------------------------
Electrical_Conductivity
0.11    124
0.12     55
0.13     33
0.14     50
0.15     67
       ... 
3.46     45
3.47     62
3.48     67
3.49     70
3.50      3
Name: count, Length: 339, dtype: int64
--------------------------
Temperature_C
12.01     2
12.02    36


In [271]:
train_data.describe()

,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,Previous_Irrigation_mm
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,6.482147,37.302524,0.923427,1.752603,26.963501,61.391652,1467.734370,7.494555,10.420977,7.555301,62.320637
std,0.921514,16.427186,0.367436,0.954866,8.629816,19.636499,611.374261,1.995983,5.720879,4.213710,34.290409
min,4.800000,8.020000,0.300000,0.110000,12.010000,25.000000,2.200000,4.000000,0.500000,0.310000,0.020000
25%,5.690000,22.940000,0.600000,0.930000,19.470000,45.310000,968.627500,5.750000,5.200000,3.920000,32.802500
50%,6.440000,37.990000,0.910000,1.740000,26.940000,61.520000,1471.250000,7.580000,10.540000,7.450000,61.180000
75%,7.270000,51.510000,1.230000,2.590000,34.540000,78.890000,2060.312500,9.220000,15.560000,11.160000,92.775000
max,8.200000,64.990000,1.600000,3.500000,41.990000,94.980000,2499.690000,10.990000,20.000000,15.000000,119.990000


In [272]:
train_data.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


## adding features 


In [320]:
def FeatureAddition(df) : 
    df['moisture_per_ph'] = df['Soil_Moisture'] / df['Soil_pH'] 
    df['rainfall_per_area'] = (df['Rainfall_mm'] /1000) * df['Field_Area_hectare'] 
    df['temp_humd_comp'] = np.sqrt((df['Humidity']**2 + df['Temperature_C']**2))
    df['field_chemical_features'] = np.sqrt((df['Organic_Carbon']**2 + df['Electrical_Conductivity']**2))
    df['head_exposure'] = np.sqrt((df['Sunlight_Hours']**2 + df['Temperature_C']**2))
    return df 

attr_adder = FunctionTransformer(FeatureAddition, validate=False) 
train_data = attr_adder.fit_transform(train_data)
test_data = attr_adder.transform(test_data)

In [321]:
# custom Cat ENcoder 
class Cat_Encoder(BaseEstimator , TransformerMixin) : 
    def __init__(self , cat_cols) : 
        self.cat_cols = cat_cols 
        self.maps_  = {}
    def fit(self , X , y =None) : 
        if not self.cat_cols : 
            self.cat_cols = X.select_dtypes(include=['category']).columns.tolist()
        self.maps_= {} 
        for col in self.cat_cols :
            
            self.maps_[col] = {cat : index for index , cat in enumerate(X.groupby(col)[col].value_counts().index)} 
        return self 
    def transform(self ,X ) : 
        for col in self.cat_cols : 
            if col in X.columns.to_list() :
                X[col] = X[col].map(self.maps_[col])
            else :  pass 
        return X

In [322]:
cat_encoder = Cat_Encoder(CAT_COLS) 
train_data = cat_encoder.fit_transform(train_data) 
test_data = cat_encoder.transform(test_data) 

In [323]:
## adding combinations of features 
from itertools import combinations
# columns_to_encode = test.columns.to_list()

pair_size = [2 ]


class FeatureConcatination(BaseEstimator , TransformerMixin) : 
    def __init__(self , columns_to_encode , pair_size = [2 ,3 ] , Encoded_cols = []) : 
        self.columns_to_encode = columns_to_encode
        self.pair_size = pair_size 
        self.Encoded_cols = []
    def fit(self , X , y= None) : 
        return self 
    def transform(self , X  ) : 
        self.Encoded_cols = []
        for r in self.pair_size : 
            combinations_list = list(combinations(self.columns_to_encode,r)) 
            for i in range(len(combinations_list)):
                new_col_name = '_'.join(combinations_list[i])
                self.Encoded_cols.append(new_col_name)
                X[new_col_name] = X[list(combinations_list[i])].astype(str).agg('_'.join, axis=1) 
                X[new_col_name] = X[new_col_name].astype('category')
        return X 

    

In [324]:
cols = CAT_COLS[:-1]
feature_cocat = FeatureConcatination(cols) 
train_data = feature_cocat.transform(train_data) 
test_data = feature_cocat.transform(test_data)

In [325]:
ENCODED_COLUMNS = feature_cocat.get_params()['Encoded_cols']

In [313]:
class TargetEncoding(BaseEstimator , TransformerMixin) : 
    def __init__(self , encoded_columns , TARGET ) : 
        self.encoded_columns = encoded_columns 
        self.TARGET = TARGET 
        self.maps_ =  {} 
    def fit(self , X , y= None) :
        self.maps_ = {}

        for col in self.encoded_columns:

            m = X.groupby(col)[self.TARGET].mean().to_dict()
            self.maps_[col] = m
        return self
    def transform(self , X ) : 
        X = X.copy()

        for col in self.encoded_columns:

            X[col] = X[col].map(self.maps_[col])
        return X 

In [43]:
## example of Target encoding 
# train_data['Soil_Type_Season'] = train_data['Soil_Type'].astype('str') + '_' + train_data['Season'].astype('str')
# train_data['Soil_Type_Humidity'] = train_data['Soil_Type'].astype('str') + '_' + train_data['Humidity'].astype('str')
# train_data['Rainfall_mm_Wind_Speed_kmh'] = train_data['Rainfall_mm'].astype('str') + '_' + train_data['Wind_Speed_kmh'].astype('str')
# print(train_data.groupby('Soil_Type_Season')[TARGET].mean().index)
# print(20*'-')
# m  =
# print(train_data.groupby('Soil_Type_Season')[TARGET].mean().to_dict())
#     print(type(i)) 
#     print(j)
# print(train_data.groupby('Soil_Type_Humidity')[TARGET].mean())
# print(20*'-')
# print(train_data.groupby('Rainfall_mm_Wind_Speed_kmh')[TARGET].mean())

In [264]:
train_data.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,moisture_per_ph,rainfall_per_area,temp_humd_comp,field_chemical_features,head_exposure
id,,,,,,,,,,,,,,,,,,,,,
0,1,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,...,0.82,0,112.16,1,1,6.621951,0.595312,52.788940,3.212880,16.127929
1,0,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,...,5.27,1,47.16,3,1,7.995763,5.194428,71.626154,2.047828,23.959274
2,0,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,...,8.24,1,110.38,2,1,4.869947,18.142008,96.082825,2.943637,27.640250
3,2,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,...,8.32,1,53.85,3,2,2.357522,11.292986,62.994343,1.589277,16.143011
4,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,...,7.37,0,93.19,3,1,7.429648,11.336534,93.326741,1.032473,21.381087


In [339]:
train_data

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Season_Irrigation_Type_Water_Source,Season_Irrigation_Type_Mulching_Used,Season_Irrigation_Type_Region,Season_Water_Source_Mulching_Used,Season_Water_Source_Region,Season_Mulching_Used_Region,Irrigation_Type_Water_Source_Mulching_Used,Irrigation_Type_Water_Source_Region,Irrigation_Type_Mulching_Used_Region,Water_Source_Mulching_Used_Region
id,,,,,,,,,,,,,,,,,,,,,
0,1,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,...,2_1_1,2_1_0,2_1_1,2_1_0,2_1_1,2_0_1,1_1_0,1_1_1,1_0_1,1_0_1
1,0,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,...,0_2_3,0_2_1,0_2_3,0_3_1,0_3_3,0_1_3,2_3_1,2_3_3,2_1_3,3_1_3
2,0,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,...,0_3_2,0_3_1,0_3_2,0_2_1,0_2_2,0_1_2,3_2_1,3_2_2,3_1_2,2_1_2
3,2,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,...,0_0_3,0_0_1,0_0_3,0_3_1,0_3_3,0_1_3,0_3_1,0_3_3,0_1_3,3_1_3
4,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,...,1_0_3,1_0_0,1_0_3,1_3_0,1_3_3,1_0_3,0_3_0,0_3_3,0_0_3,3_0_3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,0,6.54,13.45,1.15,1.86,26.65,26.86,1041.33,10.62,18.85,...,0_3_3,0_3_0,0_3_3,0_3_0,0_3_3,0_0_3,3_3_0,3_3_3,3_0_3,3_0_3
629996,0,7.03,54.49,0.96,2.35,36.99,88.00,1419.57,9.93,17.99,...,0_1_0,0_1_1,0_1_0,0_0_1,0_0_0,0_1_0,1_0_1,1_0_0,1_1_0,0_1_0
629997,0,6.52,11.98,0.93,0.38,37.82,70.98,88.45,8.19,17.25,...,2_0_2,2_0_1,2_0_3,2_2_1,2_2_3,2_1_3,0_2_1,0_2_3,0_1_3,2_1_3


In [378]:
train_data.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Season_Irrigation_Type_Water_Source,Season_Irrigation_Type_Mulching_Used,Season_Irrigation_Type_Region,Season_Water_Source_Mulching_Used,Season_Water_Source_Region,Season_Mulching_Used_Region,Irrigation_Type_Water_Source_Mulching_Used,Irrigation_Type_Water_Source_Region,Irrigation_Type_Mulching_Used_Region,Water_Source_Mulching_Used_Region
id,,,,,,,,,,,,,,,,,,,,,
0,1,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,...,2_1_1,2_1_0,2_1_1,2_1_0,2_1_1,2_0_1,1_1_0,1_1_1,1_0_1,1_0_1
1,0,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,...,0_2_3,0_2_1,0_2_3,0_3_1,0_3_3,0_1_3,2_3_1,2_3_3,2_1_3,3_1_3
2,0,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,...,0_3_2,0_3_1,0_3_2,0_2_1,0_2_2,0_1_2,3_2_1,3_2_2,3_1_2,2_1_2
3,2,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,...,0_0_3,0_0_1,0_0_3,0_3_1,0_3_3,0_1_3,0_3_1,0_3_3,0_1_3,3_1_3
4,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,...,1_0_3,1_0_0,1_0_3,1_3_0,1_3_3,1_0_3,0_3_0,0_3_3,0_0_3,3_0_3


In [390]:
train_data.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Season_Irrigation_Type_Water_Source,Season_Irrigation_Type_Mulching_Used,Season_Irrigation_Type_Region,Season_Water_Source_Mulching_Used,Season_Water_Source_Region,Season_Mulching_Used_Region,Irrigation_Type_Water_Source_Mulching_Used,Irrigation_Type_Water_Source_Region,Irrigation_Type_Mulching_Used_Region,Water_Source_Mulching_Used_Region
id,,,,,,,,,,,,,,,,,,,,,
0,1,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,...,2_1_1,2_1_0,2_1_1,2_1_0,2_1_1,2_0_1,1_1_0,1_1_1,1_0_1,1_0_1
1,0,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,...,0_2_3,0_2_1,0_2_3,0_3_1,0_3_3,0_1_3,2_3_1,2_3_3,2_1_3,3_1_3
2,0,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,...,0_3_2,0_3_1,0_3_2,0_2_1,0_2_2,0_1_2,3_2_1,3_2_2,3_1_2,2_1_2
3,2,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,...,0_0_3,0_0_1,0_0_3,0_3_1,0_3_3,0_1_3,0_3_1,0_3_3,0_1_3,3_1_3
4,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,...,1_0_3,1_0_0,1_0_3,1_3_0,1_3_3,1_0_3,0_3_0,0_3_3,0_0_3,3_0_3


# using just one model

In [ ]:
import gc
gc.collect()
N_SPLITS = 5 
skf = StratifiedKFold(n_splits = N_SPLITS , shuffle = True , random_state = 42)

oof = np.zeros((len(train_data)))
y = train_data[TARGET] 
X = train_data.drop(columns=[TARGET])
sample_weights = compute_sample_weight('balanced', y)
for k , (train_idx , val_idx) in enumerate(  skf.split(X, y )) : 
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X.iloc[val_idx], y.iloc[val_idx]
    
    encoder = TargetEncoding(ENCODED_COLUMNS , TARGET ) 
    X_train = encoder.fit_transform(pd.concat([pd.concat([X_train, y_train] , axis = 1)]) ) 
    X_train = X_train.drop(columns=[TARGET])
    X_valid = encoder.transform(X_valid)
    wt = sample_weights[train_idx]
    model = XGBClassifier(tree_method='hist',
        max_depth=14,
        colsample_bytree=0.5,
        subsample=0.9,
        n_estimators=50_000,
        learning_rate=0.02,
        enable_categorical=True,
        min_child_weight=10,
        early_stopping_rounds=150)
    model.fit(X_train , y_train ,eval_set=   [(X_valid , y_valid)] , sample_weight= wt  ,verbose =500)  
    oof[val_idx]  = model.predict(X_valid)
    
acc = accuracy_score(y, oof)


/var/folders/nk/4c5sy4tn14bgl64g5cs3dmwm0000gn/T/ipykernel_23384/632122515.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  m = X.groupby(col)[self.TARGET].mean().to_dict()


[0]	validation_0-mlogloss:1.08218
[500]	validation_0-mlogloss:0.34335
[1000]	validation_0-mlogloss:0.33920
[1023]	validation_0-mlogloss:0.33937


/var/folders/nk/4c5sy4tn14bgl64g5cs3dmwm0000gn/T/ipykernel_23384/632122515.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  m = X.groupby(col)[self.TARGET].mean().to_dict()


[0]	validation_0-mlogloss:1.08220
[500]	validation_0-mlogloss:0.34362
[917]	validation_0-mlogloss:0.34008


/var/folders/nk/4c5sy4tn14bgl64g5cs3dmwm0000gn/T/ipykernel_23384/632122515.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  m = X.groupby(col)[self.TARGET].mean().to_dict()


[0]	validation_0-mlogloss:1.08218
[500]	validation_0-mlogloss:0.34731
[919]	validation_0-mlogloss:0.34346


/var/folders/nk/4c5sy4tn14bgl64g5cs3dmwm0000gn/T/ipykernel_23384/632122515.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  m = X.groupby(col)[self.TARGET].mean().to_dict()


[0]	validation_0-mlogloss:1.08226
[500]	validation_0-mlogloss:0.34723
[920]	validation_0-mlogloss:0.34336


/var/folders/nk/4c5sy4tn14bgl64g5cs3dmwm0000gn/T/ipykernel_23384/632122515.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  m = X.groupby(col)[self.TARGET].mean().to_dict()


[0]	validation_0-mlogloss:1.08225
[500]	validation_0-mlogloss:0.34576
[1000]	validation_0-mlogloss:0.34201
[1013]	validation_0-mlogloss:0.34211


In [405]:
acc

0.8327126984126985

In [400]:
score = balanced_accuracy_score(y, oof)

In [401]:
score

0.4265956114264386

# using multiple models 

In [377]:
# 3 models, GPU-accelerated, shallow trees for speed
# LightGBM
lgb_params = dict(
     n_estimators=500, learning_rate=0.1,
    num_leaves=31, max_depth=6, class_weight='balanced', verbose=-1
)
# XGBoost
xgb_params = dict(
     tree_method='hist', enable_categorical=True,
    n_estimators=500, learning_rate=0.1, max_depth=6
)
# CatBoost (shallow & fast like the 0.967 notebook)

model_configs = [
    ('lgb', lgm.LGBMClassifier, lgb_params, False),
    ('xgb', XGBClassifier, xgb_params, True),
]

print(f'{len(model_configs)} model configs ready (GPU, ~5 min total)')

2 model configs ready (GPU, ~5 min total)


In [ ]:
import gc
gc.collect()
N_SPLITS = 5 
skf = StratifiedKFold(n_splits = N_SPLITS , shuffle = True , random_state = 42)

oof_preds = {name : np.zeros((len(train_data) , 3)) for  name , _, _, _ in model_configs }
y = train_data[TARGET] 
X = train_data.drop(columns=[TARGET])
sample_weights = compute_sample_weight('balanced', y)
for k , (train_idx , val_idx) in enumerate(  skf.split(X, y )) : 
    X_train, y_train = X.iloc[idx_train], y.iloc[idx_train]
    X_valid, y_valid = X.iloc[idx_valid], y.iloc[idx_valid]
    
    encoder = TargetEncoding(ENCODED_COLUMNS , TARGET ) 
    X_train = encoder.fit_transform(pd.concat([pd.concat([X_train, y_train] , axis = 1)]) ) 
    X_train = X_train.drop(columns=[TARGET])
    X_valid = encoder.transform(X_valid)
    wt = sample_weights[train_idx]
    for name  , model , params , swd in model_configs : 
        m = model(**params) 
        if swd : 
            m.fit(X_train , y_train , sample_weight =wt)
        else : 
            m.fit(X_train , y_train) 
        oof_preds[name][val_idx]  = m.predict_proba(X_valid)
        score = balanced_accuracy_score(y_valid, oof_preds[name][val_idx].argmax(1))
        print(f'  {name}: {score:.5f}')
       
    
print(f'\n=== Overall OOF Balanced Accuracy ===')
for name in oof_preds:
    score = balanced_accuracy_score(y, oof_preds[name].argmax(1))
    print(f'  {name}: {score:.5f}')


In [388]:
oof_stack = np.hstack([oof_preds[name] for name in oof_preds])
avg_oof = np.mean([oof_preds[name] for name in oof_preds], axis=0)
avg_score = balanced_accuracy_score(y, avg_oof.argmax(1))
print(f'Simple average OOF balanced accuracy: {avg_score:.5f}')
meta_oof_preds = np.zeros(len(y) , dtype = int) 
for fold , (train_idx , val_idx) in enumerate(skf.split(X, y) ) : 
    meta = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0, random_state=42)
    meta.fit(oof_stack[train_idx], y[train_idx])
    meta_oof_preds[val_idx] = meta.predict(oof_stack[val_idx])


stack_score = balanced_accuracy_score(y, meta_oof_preds)
print(f'Stacked meta-learner OOF balanced accuracy: {stack_score:.5f}')

# Also train final meta-model on all data
final_meta = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0, random_state=42)
final_meta.fit(oof_stack, y)
# final_meta_test = final_meta.predict_proba(test_stack)

# Pick best approach
if stack_score >= avg_score:
    print('\n>>> Using stacked meta-learner predictions')
#     final_test_probs = final_meta_test
    best_oof_probs = np.zeros((len(y), 3))
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
        meta = LogisticRegression(class_weight='balanced', max_iter=2000, C=1.0, random_state=42)
        meta.fit(oof_stack[tr_idx], y[tr_idx])
        best_oof_probs[val_idx] = meta.predict_proba(oof_stack[val_idx])
else:
    print('\n>>> Using simple average predictions')
#     final_test_probs = avg_test
    best_oof_probs = avg_oof

Simple average OOF balanced accuracy: 0.33379
Stacked meta-learner OOF balanced accuracy: 0.33341

>>> Using simple average predictions
